# Regression examples

In [2]:
from sklearn.datasets import load_diabetes
import itertools
import sys

sys.path.append("..")

from ai_toolkit import (
    BaseDataset, 
    MlTrainerConfig,
    RidgeRegressionModel,
    KNNRegressorModel,
    LightGBMRegressorModel, 
    get_all_regression_models, 
    RegressionModelTrainer, 
    lazypredict_regression,
    EnsembleVotingRegressorModel,
    EnsembleStackingRegressorModel,
)

## Example data

In [3]:
class RegDataset(BaseDataset):
    """Dataset for diabetes regression task.
    https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html
    """
    def __init__(self):
        """Initialize the clf dataset."""

        super().__init__()

    def load_data(self):
        """Load the diabetes dataset for regression task."""
        
        self.X, self.y = load_diabetes(return_X_y=True, as_frame=True)
        self.X_test = self.X.head()

In [4]:
RDataset = RegDataset()
RDataset.load_data()
RDataset.preprocess()
X_reg, y_reg, X_test_reg = RDataset.get_data()

## Configuration

In [5]:
TrainerConfig = MlTrainerConfig()
TrainerConfig.N_TRAILS = 2
TrainerConfig.EXPERIMENT_NAME = "ml_regression"
TrainerConfig.OPTIMIZE_METRIC = "root_mean_squared_error"

## Training and evaluation

### Train one example model

In [6]:
base_model = RidgeRegressionModel()

# Create a regression model trainer
trainer = RegressionModelTrainer(
    base_model=base_model,
    config=TrainerConfig,
)

In [7]:
# Train and optimize the model
best_model, mean_metrics = trainer.train_and_optimize(
    X=X_reg, 
    y=y_reg, 
    n_trials=TrainerConfig.N_TRAILS, 
)

# y_pred = trainer.predict(X_test)

[I 2025-04-03 14:11:42,195] A new study created in memory with name: Ridge Regressor optimization
[I 2025-04-03 14:11:42,411] Trial 0 finished with value: 165.8333303236106 and parameters: {'alpha': 6.186082659867516e-05, 'fit_intercept': False, 'solver': 'svd', 'tol': 0.0001388304808143499, 'random_state': 28}. Best is trial 0 with value: 165.8333303236106.
[I 2025-04-03 14:11:42,584] Trial 1 finished with value: 55.268883987281924 and parameters: {'alpha': 75.12774418058405, 'fit_intercept': True, 'solver': 'lsqr', 'tol': 6.169008786817236e-05, 'random_state': 28}. Best is trial 0 with value: 165.8333303236106.
Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/89 [00:00<?, ?it/s]

Cross-validation:  80%|████████  | 4/5 [00:59<00:08,  8.59s/it]

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation: 100%|██████████| 5/5 [01:48<00:00, 21.61s/it]



# Model: Ridge Regressor

## Best Hyperparameters: {'alpha': 6.186082659867516e-05, 'fit_intercept': False, 'solver': 'svd', 'tol': 0.0001388304808143499, 'random_state': 28}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 170.1165
Fold 2 Score: 165.4887
Fold 3 Score: 163.0720
Fold 4 Score: 158.9858
Fold 5 Score: 171.5037

## Mean Metrics across all folds:
d2_pinball: -1.4611
mean_squared_error: 27521.7198
d2_tweedie: -3.8282
max_error: 283.3418
r2: -3.8282
mean_absolute_percentage_error: 1.2107
mean_absolute_error: 156.0652
explained_variance: 0.4425
median_absolute_error: 151.6840
root_mean_squared_error: 165.8333
d2_absolute_error: -1.4611


### Train all regression models

In [8]:
base_models = get_all_regression_models()

for base_model in base_models.values():

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
        n_trials=TrainerConfig.N_TRAILS,
    )

[I 2025-04-03 14:13:44,345] A new study created in memory with name: Ridge Regressor optimization
[I 2025-04-03 14:13:44,397] Trial 0 finished with value: 55.107470390628905 and parameters: {'alpha': 0.016330682610915276, 'fit_intercept': True, 'solver': 'lsqr', 'tol': 0.00022337953842686072, 'random_state': 28}. Best is trial 0 with value: 55.107470390628905.
[I 2025-04-03 14:13:44,461] Trial 1 finished with value: 55.02209423999216 and parameters: {'alpha': 5.0639410755978895, 'fit_intercept': True, 'solver': 'lsqr', 'tol': 3.4435796961739364e-05, 'random_state': 28}. Best is trial 0 with value: 55.107470390628905.
Cross-validation:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/89 [00:00<?, ?it/s]

Cross-validation:  60%|██████    | 3/5 [00:54<00:25, 12.70s/it]

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation: 100%|██████████| 5/5 [01:47<00:00, 21.60s/it]


...

  0%|          | 0/89 [00:00<?, ?it/s]

Cross-validation:  60%|██████    | 3/5 [01:53<00:51, 25.97s/it] 

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation: 100%|██████████| 5/5 [03:34<00:00, 42.94s/it]



# Model: CatBoost Regressor

## Best Hyperparameters: {'iterations': 305, 'learning_rate': 0.08270788126783635, 'depth': 4, 'l2_leaf_reg': 1.386498103204251e-06, 'bootstrap_type': 'Bayesian', 'random_strength': 5.989541717883e-08, 'bagging_temperature': 2.935459441224551, 'od_type': 'Iter', 'od_wait': 44, 'verbose': False, 'random_seed': 28, 'task_type': 'CPU'}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 60.7656
Fold 2 Score: 51.1386
Fold 3 Score: 54.5634
Fold 4 Score: 65.5701
Fold 5 Score: 55.4944

## Mean Metrics across all folds:
d2_pinball: 0.2708
mean_squared_error: 3332.7687
d2_tweedie: 0.4073
max_error: 163.0047
r2: 0.4073
mean_absolute_percentage_error: 0.4027
mean_absolute_error: 46.1045
explained_variance: 0.4133
median_absolute_error: 41.3148
root_mean_squared_error: 57.5064
d2_absolute_error: 0.2708


In [9]:
df_mean_results = lazypredict_regression(
    X=X_reg, 
    y=y_reg, 
    n_splits=TrainerConfig.N_SPLITS,
    random_state=TrainerConfig.RANDOM_STATE,
)

# print(df_mean_results.to_string())
df_mean_results

Cross-Validation: 100%|██████████| 5/5 [00:26<00:00,  5.35s/it]


,Adjusted R-Squared,R-Squared,RMSE,Time Taken,Adjusted R-Squared Std
Model,,,,,
PoissonRegressor,0.40,0.47,54.79,0.03,0.14
LassoLars,0.39,0.46,54.95,0.03,0.14
Lasso,0.39,0.46,54.95,0.03,0.14
ElasticNetCV,0.39,0.46,55.06,0.15,0.13
BayesianRidge,0.39,0.46,55.04,0.03,0.13
SGDRegressor,0.39,0.46,55.06,0.02,0.14
LassoLarsIC,0.39,0.46,55.07,0.03,0.14
Ridge,0.39,0.46,55.06,0.02,0.14
LinearRegression,0.39,0.46,55.11,0.03,0.15


### Ensemble

In [10]:
# (Model, mlflow run_id) pairs
model_pool = [
    (RidgeRegressionModel(), "8176863ad9b344c6be3f00f9c5987737"),
    (KNNRegressorModel(), "41444ec16eba4a60b46aff3ad933e178"),
    (LightGBMRegressorModel(), "db8511fbb32b4582be011d3efb1daec4"),
]

meta_model = RidgeRegressionModel()

In [11]:
combinations = []
 
for model in range(2, len(model_pool) + 1):
    combinations.extend(itertools.combinations(model_pool, model))

#### Voting | Train all combinations

In [12]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleVotingRegressorModel(
        models=models,
    )

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )
    
    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
        n_trials=TrainerConfig.N_TRAILS, 
    )

    # y_pred = trainer.predict(X_test)

[I 2025-04-03 14:41:40,142] A new study created in memory with name: Get best parameters from MLflow
[I 2025-04-03 14:41:40,248] A new study created in memory with name: Get best parameters from MLflow
[I 2025-04-03 14:41:40,644] A new study created in memory with name: Ensemble Voting Regressor optimization
[I 2025-04-03 14:41:40,731] Trial 0 finished with value: 54.917346761779655 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=0.016330682610915276, random_state=28, solver='lsqr',
      tol=0.00022337953842686072)), ('K-Nearest Neighbors Regressor', KNeighborsRegressor(leaf_size=13, metric='euclidean', n_neighbors=35, p=1,
                    weights='distance'))], 'weight_0': 0.9253520400589246, 'weight_1': 0.5141082201995018}. Best is trial 0 with value: 54.917346761779655.
[I 2025-04-03 14:41:40,815] Trial 1 finished with value: 55.726320957394606 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=0.016330682610915276, random_state=28, solver='lsqr',
 

  0%|          | 0/89 [00:00<?, ?it/s]

Cross-validation:  60%|██████    | 3/5 [05:32<02:30, 75.05s/it] 

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation:  80%|████████  | 4/5 [09:43<02:24, 144.63s/it]

...

Cross-validation:  80%|████████  | 4/5 [03:27<00:28, 28.69s/it]

[LightGBM] [Warning] feature_fraction is set=0.6454880943659053, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6454880943659053
[LightGBM] [Warning] lambda_l1 is set=1.0494770613699195e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0494770613699195e-07
[LightGBM] [Warning] bagging_fraction is set=0.7415879653318771, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7415879653318771
[LightGBM] [Warning] lambda_l2 is set=3.628649053447043e-08, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.628649053447043e-08
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3


  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation: 100%|██████████| 5/5 [06:47<00:00, 81.43s/it]



# Model: Ensemble Voting Regressor

## Best Hyperparameters: {'estimators': [('Ridge Regressor', Ridge(alpha=0.016330682610915276, random_state=28, solver='lsqr',
      tol=0.00022337953842686072)), ('K-Nearest Neighbors Regressor', KNeighborsRegressor(leaf_size=13, metric='euclidean', n_neighbors=35, p=1,
                    weights='distance')), ('LightGBM Regressor', LGBMRegressor(bagging_fraction=0.7415879653318771, bagging_freq=3,
              boosting_type='dart', device_type='cpu',
              feature_fraction=0.6454880943659053,
              lambda_l1=1.0494770613699195e-07, lambda_l2=3.628649053447043e-08,
              learning_rate=0.05019567565880687, metric='rmse',
              min_child_samples=71, num_leaves=22, objective='regression',
              seed=28, verbose=-1))], 'weights': [0.12713026851837517, 0.047147217627237614, 0.32893751644534597]}

## Optimize metric 'root_mean_squared_error' for each fold:
Fold 1 Score: 61.2165
Fold 2 Score: 55.3840
Fold 3 Score:

#### Stacking | Train all combinations

In [13]:
for combination in combinations:
    models = list(combination)

    base_model = EnsembleStackingRegressorModel(
        models=models,
        meta_model=meta_model,
    )

    # Create a regression model trainer
    trainer = RegressionModelTrainer(
        base_model=base_model,
        config=TrainerConfig,
    )

    # Train and optimize the model
    best_model, mean_metrics = trainer.train_and_optimize(
        X=X_reg, 
        y=y_reg, 
        n_trials=TrainerConfig.N_TRAILS, 
    )

    # y_pred = trainer.predict(X_test)

[I 2025-04-03 15:12:28,241] A new study created in memory with name: Get best parameters from MLflow
[I 2025-04-03 15:12:28,374] A new study created in memory with name: Get best parameters from MLflow
[I 2025-04-03 15:12:28,707] A new study created in memory with name: Ensemble Stacking Regressor optimization
[I 2025-04-03 15:12:28,995] Trial 0 finished with value: 55.04744180978298 and parameters: {'estimators': [('Ridge Regressor', Ridge(alpha=0.016330682610915276, random_state=28, solver='lsqr',
      tol=0.00022337953842686072)), ('K-Nearest Neighbors Regressor', KNeighborsRegressor(leaf_size=13, metric='euclidean', n_neighbors=35, p=1,
                    weights='distance'))], 'alpha': 0.22075443712313514, 'fit_intercept': False, 'solver': 'auto', 'tol': 3.5067782895173342e-06, 'random_state': 28, 'passthrough': True}. Best is trial 0 with value: 55.04744180978298.
[I 2025-04-03 15:12:29,296] Trial 1 finished with value: 54.97387240072468 and parameters: {'estimators': [('Ridge 

  0%|          | 0/89 [00:00<?, ?it/s]

Cross-validation:  60%|██████    | 3/5 [03:20<01:30, 45.44s/it] 

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation: 100%|██████████| 5/5 [06:33<00:00, 78.78s/it] 


...

  0%|          | 0/88 [00:00<?, ?it/s]

Cross-validation:  80%|████████  | 4/5 [06:50<01:48, 108.02s/it]

[LightGBM] [Warning] feature_fraction is set=0.6454880943659053, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6454880943659053
[LightGBM] [Warning] lambda_l1 is set=1.0494770613699195e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0494770613699195e-07
[LightGBM] [Warning] bagging_fraction is set=0.7415879653318771, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7415879653318771
[LightGBM] [Warning] lambda_l2 is set=3.628649053447043e-08, reg_lambda=0.0 will be ignored. Current value: lambda_l2=3.628649053447043e-08
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6454880943659053, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6454880943659053
[LightGBM] [Warning] lambda_l1 is set=1.0494770613699195e-07, reg_alpha=0.0 will be ignored. Current value: lambda_l1=1.0494770613699195e-07
[LightGBM] [War

Cross-validation: 100%|██████████| 5/5 [06:53<00:00, 82.79s/it] 



# Model: Ensemble Stacking Regressor

## Best Hyperparameters: {'estimators': [('Ridge Regressor', Ridge(alpha=0.016330682610915276, random_state=28, solver='lsqr',
      tol=0.00022337953842686072)), ('K-Nearest Neighbors Regressor', KNeighborsRegressor(leaf_size=13, metric='euclidean', n_neighbors=35, p=1,
                    weights='distance')), ('LightGBM Regressor', LGBMRegressor(bagging_fraction=0.7415879653318771, bagging_freq=3,
              boosting_type='dart', device_type='cpu',
              feature_fraction=0.6454880943659053,
              lambda_l1=1.0494770613699195e-07, lambda_l2=3.628649053447043e-08,
              learning_rate=0.05019567565880687, metric='rmse',
              min_child_samples=71, num_leaves=22, objective='regression',
              seed=28, verbose=-1))], 'alpha': 82.18984664665102, 'fit_intercept': False, 'solver': 'svd', 'tol': 1.6061732587725793e-06, 'random_state': 28, 'passthrough': False}

## Optimize metric 'root_mean_squared_error' for e